# GC-LSTM-GhostNet â€” Step 2 smoke test

This notebook checks the preserved CIC-DDoS2019 Parquet schema, group-first 70/80 splits, leakage assertions, and train-only preprocessing. It does not train the final model.

In [ ]:
from pathlib import Path
import base64
import io
import json
import os
import shutil
import subprocess
import sys
import zipfile
from kaggle_secrets import UserSecretsClient

PROJECT_DIR = Path("/kaggle/working/Luan-Van-GC-LSTM-GhostNet-CICDDoS2019-v1")
OUTPUT_DIR = PROJECT_DIR / "outputs" / "step2_smoke"
DATA_DIR = Path("/kaggle/working/cicddos2019-parquet-input")
PROJECT_ARCHIVE_B64 = "..."

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ARCHIVE_B64))) as archive:
    archive.extractall(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True)
download_env = os.environ.copy()
secret_value = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
try:
    classic = json.loads(secret_value)
except (TypeError, json.JSONDecodeError):
    download_env["KAGGLE_API_TOKEN"] = secret_value
else:
    download_env["KAGGLE_USERNAME"] = classic["username"]
    download_env["KAGGLE_KEY"] = classic["key"]
subprocess.run(["kaggle", "datasets", "download", "-d", "dungnguyen28101991/cicddos2019-parquet", "-p", str(DATA_DIR), "--unzip", "--quiet"], env=download_env, check=True)
for key in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
    download_env.pop(key, None)
del secret_value
print(f"Project and private dataset ready; dataset files: {sum(1 for p in DATA_DIR.rglob('*') if p.is_file())}")

In [ ]:
command = [
    sys.executable, "-m", "src.step2_smoke",
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--config", "configs/base.yaml",
    "--mode-config", "configs/practical_baseline.yaml",
    "--samples-per-file", "2048",
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)

In [ ]:
summary_path = OUTPUT_DIR / "smoke_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary["status"] == "passed", summary
assert all(run["leakage_status"] == "passed" for run in summary["runs"]), summary
summary